# 02 — Why one neuron is not enough: the hidden layer

In the last notebook a single neuron turned out to be logistic regression — and it draws a
**straight** boundary. That is a real limit. Here we meet the idea that lifts it: a **hidden layer**,
and the non-linearity that makes it work.

**Prerequisites**
- 11.1 — the artificial neuron == logistic regression; the activations sigmoid / tanh / ReLU.
- Chapter 00 — accuracy and the train/test split.

**What you'll be able to do**
- Show, on XOR and concentric circles, what a single straight boundary cannot do.
- Build a `2 → H → 1` network's forward pass by hand.
- Explain why a non-linear activation is essential — a stack of linear layers collapses to one.
- See a hidden layer with a non-linearity solve both problems.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from ml_course import colors, viz

viz.use_course_style()

SEED = 0
np.random.seed(SEED)

## Where we left off

In notebook 11.1 we found that an artificial neuron is `a = φ(w·x + b)`, and that a single sigmoid
neuron is exactly logistic regression, whose decision boundary is a **straight line** (a hyperplane
in higher dimensions). We also met three activations `φ` — the S-shaped **sigmoid** and **tanh**, and
**ReLU** (`max(0, z)`) — the pluggable non-linearities we reach for again here. That straight boundary
is the whole story for one neuron, so the question is: what can a straight line *not* separate?

## Two problems no straight line can split

- **XOR** — four points: the output is 1 when *exactly one* of the two inputs is on. The two "1"
  corners sit on opposite diagonals, so no single line puts both on one side.
- **Concentric circles** — an inner disc of one class surrounded by an outer ring of the other. No
  straight cut keeps the disc on one side and the ring on the other.

Let's watch a single neuron (logistic regression) try.

In [ ]:
# XOR: four corner points.
X_xor = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = np.array([0, 1, 1, 0])

# Concentric circles, split into train / test.
X_circ, y_circ = make_circles(n_samples=400, noise=0.10, factor=0.40, random_state=SEED)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_circ, y_circ, test_size=0.25, stratify=y_circ, random_state=SEED
)

log_xor = LogisticRegression().fit(X_xor, y_xor)
log_circ = LogisticRegression().fit(Xc_tr, yc_tr)
print(f"XOR      logistic accuracy = {log_xor.score(X_xor, y_xor):.2f}")
print(f"circles  logistic test acc = {log_circ.score(Xc_te, yc_te):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
viz.plot_decision_boundary(log_xor, X_xor, y_xor, ax=axes[0])
axes[0].set_title(f"XOR — one straight line (acc {log_xor.score(X_xor, y_xor):.2f})")
viz.plot_decision_boundary(log_circ, Xc_te, yc_te, ax=axes[1])
axes[1].set_title(f"Circles — one straight line (acc {log_circ.score(Xc_te, yc_te):.2f})")
plt.show()

**Read the figure.** Left: on XOR the line can keep at most one of the two "1" corners on the
positive side — accuracy 0.50, a coin flip. Right: on the circles the line slices straight through
both rings — accuracy 0.41, a hair under 0.50 and, on 100 balanced test points, squarely within the
coin-flip range: a straight cut through concentric rings carries essentially no signal. A single
neuron is one straight cut, and neither problem yields to a straight cut. We need something that bends.

## Stacking neurons: a hidden layer

Instead of one neuron, place `H` of them side by side — a **hidden layer**. Each hidden neuron has
its own weights, bias, and activation, so each computes its own `φ(wⱼ·x + bⱼ)`. Their `H` outputs
become the inputs to a final **output neuron**, which draws *its* straight line — but now in the new
space the hidden layer built. Bend the space, and a straight cut in it becomes a curved cut in the
original.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axis("off")
C = colors.COLORS

inputs = [(0.0, 2.4, "x₁"), (0.0, 0.6, "x₂")]
hidden_nodes = [(2.2, hy) for hy in (3.0, 2.0, 1.0, 0.0)]
out = (4.4, 1.5)

# Edges: every input to every hidden neuron, every hidden neuron to the output.
for ix, iy, _ in inputs:
    for hx, hy in hidden_nodes:
        ax.plot([ix, hx], [iy, hy], color=C["muted"], lw=0.7, zorder=1)
for hx, hy in hidden_nodes:
    ax.plot([hx, out[0]], [hy, out[1]], color=C["muted"], lw=0.7, zorder=1)

for ix, iy, label in inputs:
    ax.add_patch(plt.Circle((ix, iy), 0.18, color=C["class_a"], ec=C["text"], lw=1.2, zorder=3))
    ax.text(ix, iy, label, ha="center", va="center", color=C["text"], zorder=4)
for hx, hy in hidden_nodes:
    ax.add_patch(plt.Circle((hx, hy), 0.20, color=C["class_c"], ec=C["text"], lw=1.2, zorder=3))
ax.add_patch(plt.Circle(out, 0.22, color=C["class_b"], ec=C["text"], lw=1.2, zorder=3))
ax.text(out[0], out[1], "ŷ", ha="center", va="center", color=C["text"], zorder=4)

ax.text(0.0, 3.3, "inputs", ha="center", color=C["muted"], fontsize=10)
ax.text(2.2, 3.7, "hidden layer (H neurons)", ha="center", color=C["muted"], fontsize=10)
ax.text(4.4, 2.1, "output", ha="center", color=C["muted"], fontsize=10)
ax.set_xlim(-0.6, 5.3)
ax.set_ylim(-0.6, 4.1)
ax.set_title("A 2 → H → 1 network")
plt.show()

**Read the figure.** Every input connects to every hidden neuron (each connection carries a weight);
the hidden neurons' outputs all feed the single output neuron `ŷ`. Two questions decide whether this
is worth anything: does adding the hidden layer actually help, and — if it does — what makes it help?

## Why the activation must be non-linear

Suppose the hidden activation is the **identity** — `φ(z) = z`, no bend. Then the hidden layer
computes `W₁x` and the output neuron computes `W₂(W₁x) = (W₂W₁)x`. The product `W₂W₁` is itself a
single weight matrix: two linear layers **collapse into one**. Stack a hundred identity layers and
you still have a single straight boundary. Without a non-linearity, depth buys nothing.

In [ ]:
# Two linear layers compose into one: (x M1) M2 == x (M1 M2).
rng = np.random.default_rng(SEED)
M1 = rng.normal(size=(2, 8))
M2 = rng.normal(size=(8, 1))
M_equiv = M1 @ M2  # a single 2 -> 1 linear map

x = rng.normal(size=(5, 2))
two_layers = (x @ M1) @ M2
one_layer = x @ M_equiv
collapse_gap = np.max(np.abs(two_layers - one_layer))
print("max |two identity layers − one linear layer| =", f"{collapse_gap:.2e}")
print("M1 @ M2 has shape", M_equiv.shape, "— the hidden layer collapsed into one neuron")

# Measured on the circles: an identity-activated hidden layer == logistic regression.
mlp_identity = MLPClassifier(
    hidden_layer_sizes=(8,), activation="identity", solver="lbfgs",
    alpha=1e-4, max_iter=5000, random_state=SEED,
).fit(Xc_tr, yc_tr)
print(f"circles  identity-MLP test acc = {mlp_identity.score(Xc_te, yc_te):.2f}")
print(f"circles  logistic    test acc = {log_circ.score(Xc_te, yc_te):.2f}")

**Read the result.** The two identity layers reproduce a single linear layer to `~2e-16` — they
*are* the same map. And on the circles, an identity-activated hidden layer of 8 neurons scores 0.41,
identical to one neuron: the hidden layer bought nothing. The non-linearity is the part that stops
the collapse.

## A hidden layer with a non-linearity, built by hand

Now give the hidden neurons a real non-linearity — **ReLU**, `max(0, z)`. Here is a tiny
`2 → 2 → 1` network whose weights we **set by hand** (the classic XOR solution). We are not training
it — finding good weights automatically is the next notebook. We only check what this fixed network
computes.

In [ ]:
# A hand-set 2 -> 2 -> 1 ReLU network (Goodfellow, Deep Learning, sec 6.1).
W1 = np.array([[1.0, 1.0], [1.0, 1.0]])
b1 = np.array([0.0, -1.0])
w2 = np.array([1.0, -2.0])
b2 = 0.0

hidden = np.maximum(0.0, X_xor @ W1 + b1)   # the ReLU hidden layer
output = hidden @ w2 + b2                    # the output neuron

print("inputs        :", X_xor.tolist())
print("network output:", output.tolist())
print("network labels:", (output > 0.5).astype(int).tolist())
print("XOR target    :", y_xor.tolist())

**Read the result.** The network's output is exactly `[0, 1, 1, 0]` — XOR, with no errors. Two ReLU
hidden units, combined by one output neuron, compute what a single neuron (0.50) cannot. We *chose*
these weights to show it is possible; the next notebook is about how a network **finds** such weights
on its own.

In [ ]:
mlp_xor = MLPClassifier(
    hidden_layer_sizes=(4,), activation="relu", solver="lbfgs",
    max_iter=5000, random_state=SEED,
).fit(X_xor, y_xor)
mlp_circ = MLPClassifier(
    hidden_layer_sizes=(8,), activation="tanh", solver="lbfgs",
    alpha=1e-4, max_iter=5000, random_state=SEED,
).fit(Xc_tr, yc_tr)
print(f"XOR      MLP(4, relu) accuracy = {mlp_xor.score(X_xor, y_xor):.2f}")
print(f"circles  MLP(8, tanh) test acc = {mlp_circ.score(Xc_te, yc_te):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
viz.plot_decision_boundary(mlp_xor, X_xor, y_xor, ax=axes[0])
axes[0].set_title(f"XOR — hidden layer + ReLU (acc {mlp_xor.score(X_xor, y_xor):.2f})")
viz.plot_decision_boundary(mlp_circ, Xc_te, yc_te, ax=axes[1])
axes[1].set_title(f"Circles — hidden layer + tanh (acc {mlp_circ.score(Xc_te, yc_te):.2f})")
plt.show()

**Read the figure.** The same `2 → H → 1` shape that collapsed under the identity now does the job:
on XOR the regions wrap around each diagonal pair (accuracy 1.0), and on the circles the boundary
closes into a loop around the inner disc (1.0). The non-linear hidden layer is the engine. We fitted
these as black boxes — *how* the weights are found is the next notebook.

## How far does a hidden layer go?

Far. A single hidden layer with enough non-linear units can approximate **any** continuous decision
boundary to arbitrary closeness (Cybenko 1989; Hornik 1991). Read that claim carefully, though: it is
an **existence** result — it says such a network *exists*, not that training will find it, and not
that it will generalize to new data. "Can represent" is not "will learn". So the honest, useful
question is the next one: given a network's shape, how do we **find** its weights? That is
**backpropagation** — notebook 11.3.

## Your turn

1. **(warm-up)** Set the circles MLP's `activation="identity"` and re-fit. Predict the test accuracy
   *before* you run it, then check — why does an 8-neuron hidden layer score the same as one neuron?
2. **(core)** Shrink the hidden layer to a single unit (`hidden_layer_sizes=(1,)`) with `tanh` on the
   circles. Does one hidden neuron help? Why is one non-linear neuron still not enough here?
3. **(reach)** A single neuron *can* learn AND and OR (both linearly separable). Hand-set or fit one
   neuron for OR on the four corner points, then explain in one sentence why XOR is the odd one out.

## What you built

- You saw a single neuron's straight-line limit on two problems it cannot solve (XOR 0.50,
  circles 0.41).
- You built a `2 → H → 1` network's forward pass by hand, and proved the **linear collapse**: without
  a non-linearity, stacked layers reduce to one (`~2e-16`; an identity hidden layer scores the same
  as one neuron).
- You hand-set a two-unit ReLU network that computes XOR exactly.
- You watched a non-linear hidden layer separate both XOR and the circles.

A hidden layer plus a non-linearity is the whole new idea. Next: how a network **learns** its weights.

## References

- Minsky, M. & Papert, S. (1969). *Perceptrons.* MIT Press — the single-unit limit (XOR).
- Cybenko, G. (1989). *Approximation by superpositions of a sigmoidal function.* Mathematics of
  Control, Signals and Systems 2, 303–314. https://doi.org/10.1007/BF02551274
- Hornik, K. (1991). *Approximation capabilities of multilayer feedforward networks.* Neural Networks
  4(2), 251–257. https://doi.org/10.1016/0893-6080(91)90009-T
- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*, §6.1 (the hand-set ReLU XOR
  network). MIT Press.

*Previous:* 11.1 — the neuron == logistic regression.  *Next:* **11.3 — how a network learns
(backpropagation).**